In [ ]:
from matplotlib_inline.backend_inline import set_matplotlib_formats
from os import listdir as ls
from IPython.display import display, Markdown
import pycountry_convert as pc

from emu_renewal.constants import OUTPUTS_PATH
from emu_renewal.utils import get_countries_by_continent
from emu_renewal.plotting import ANALYSIS_NAMES, get_idatas_for_mob_type, plot_param_posts_for_countries

In [ ]:
set_matplotlib_formats("svg")

In [ ]:
job_path = OUTPUTS_PATH / "45878514"
countries = ls(job_path)
all_countries = ls(job_path)
countries_by_cont = get_countries_by_continent(all_countries)
mob_type = "g_mob"

In [ ]:
import pycountry
import arviz as az
import pycountry
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

from emu_renewal.plotting import get_standard_subplot, MOB_COLOURS, AN_ABBREVS
from emu_renewal.utils import sort_countries_by_name

In [ ]:
mob_types = [k for k in AN_ABBREVS if k != "no_mob"]
for cont in countries_by_cont:
    i_datas = {}
    table_info = {}
    cont_name = pc.convert_continent_code_to_continent_name(cont)
    display(Markdown(f"# {cont_name}"))
    display(Markdown("## Parameter distributions"))
    countries = countries_by_cont[cont]
    for mob_type in mob_types:
        idatas, _ = get_idatas_for_mob_type(job_path, countries, mob_type)
        i_datas[mob_type] = idatas
    all_countries = sort_countries_by_name({key for subdict in i_datas.values() for key in subdict})
    fig, axes = get_standard_subplot(len(all_countries), 4)
    flat_axes = axes.ravel()
    table_info = pd.DataFrame(columns=mob_types)
    for c, country in enumerate(all_countries):
        country_name = pycountry.countries.lookup(country).name
        c_ax = flat_axes[c]
        for mob_type in mob_types:
            m_idatas = i_datas[mob_type]
            colour = MOB_COLOURS[mob_type]
            if country in m_idatas:
                c_idata = m_idatas[country]
                post_plot = az.plot_posterior(c_idata, var_names="mob_exp", ax=c_ax, point_estimate=None, hdi_prob="hide", color=colour)
                line_data = post_plot.get_lines()[-1]
                post_plot.fill_between(line_data.get_xdata(), line_data.get_ydata(), alpha=0.1, color=MOB_COLOURS[mob_type])
                mean = az.summary(c_idata, var_names="mob_exp", kind="stats")["mean"].values[0]
                table_info.loc[country_name, mob_type] = mean
        c_ax.set_title(country_name)
        c_ax.set_xlim([0.0, 2.0])
        c_ax.set_xticks(np.linspace(0.0, 2.0, 5))
        c_ax.tick_params(labelsize=9)
    stats_table = pd.DataFrame(table_info)
    stats_table = stats_table.mask(stats_table.isna(), "no analysis")
    stats_table = stats_table.rename(columns=ANALYSIS_NAMES)
    for a in range(c + 1, len(flat_axes)):
        flat_axes[a].set_axis_off()
    fig.tight_layout()
    plt.show()
    display(Markdown("## Parameter posterior means"))
    display(stats_table)